# Parte B — Fine-tuning MAE-AST su ESC-50
### Replica di Baade et al. (Interspeech 2022) — Table 1

**Obiettivo**: verificare che i checkpoint pretrainati degli autori producano risultati vicini al paper su ESC-50.

**Target paper**: Patch 90.0% | Frame 88.9%

**Protocollo**: 5-fold cross-validation (standard ufficiale ESC-50).

---

**Prima di eseguire**, assicurati di avere su Google Drive:
- `mae_ast_patch_converted.pth`
- `mae_ast_frame_converted.pth`

In [ ]:
# CELLA 0 — Verifica GPU
import torch

if torch.cuda.is_available():
    nome_gpu = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {nome_gpu}  ({vram_gb:.1f} GB VRAM)')
else:
    raise SystemExit('GPU non trovata. Runtime -> Change runtime type -> T4 GPU')

DEVICE = torch.device('cuda')
print(f'Device: {DEVICE}')

In [ ]:
# CELLA 1 — Installa dipendenze
!pip install timm==0.9.16 librosa==0.10.1 --quiet

import timm, librosa, torchaudio
print(f'timm:       {timm.__version__}')
print(f'librosa:    {librosa.__version__}')
print(f'torchaudio: {torchaudio.__version__}')
print(f'torch:      {torch.__version__}')
print('Dipendenze pronte')

In [ ]:
# CELLA 2 — Clona la repo e monta Drive
import os, sys

REPO_URL = 'https://github.com/alessiopgg/deep_learning_project.git'
REPO_DIR = '/content/mae-ast-poggesi'

if os.path.exists(REPO_DIR):
    print('Repo gia presente, aggiorno...')
    os.system(f'cd {REPO_DIR} && git pull --quiet')
else:
    print('Clono la repo...')
    os.system(f'git clone {REPO_URL} {REPO_DIR} --quiet')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Repo in: {REPO_DIR}')

from google.colab import drive
drive.mount('/content/drive')
print('Drive montato')

In [ ]:
# CELLA 3 — Configura percorsi
# ============================================================
# MODIFICA QUESTI PERCORSI con quelli reali su Drive
CHECKPOINT_PATCH = '/content/drive/MyDrive/mae_ast_patch_converted.pth'
CHECKPOINT_FRAME = '/content/drive/MyDrive/mae_ast_frame_converted.pth'
# ============================================================

RESULTS_DIR = '/content/drive/MyDrive/mae_ast_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

for nome, path in [('Patch', CHECKPOINT_PATCH), ('Frame', CHECKPOINT_FRAME)]:
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1e6
        print(f'OK {nome}: {path}  ({mb:.0f} MB)')
    else:
        print(f'MANCANTE {nome}: {path}')

print(f'Risultati in: {RESULTS_DIR}')

In [ ]:
# CELLA 4 — Scarica ESC-50
import glob

ESC50_DIR = '/content/ESC-50'
CSV_PATH  = f'{ESC50_DIR}/meta/esc50.csv'
AUDIO_DIR = f'{ESC50_DIR}/audio'

if os.path.exists(CSV_PATH):
    print('ESC-50 gia presente')
else:
    print('Scarico ESC-50 (~700 MB)...')
    os.system(f'git clone --depth 1 https://github.com/karolpiczak/ESC-50.git {ESC50_DIR} --quiet')
    print('ESC-50 scaricato')

n_file = len(glob.glob(f'{AUDIO_DIR}/*.wav'))
print(f'File audio: {n_file}/2000')

import pandas as pd
df = pd.read_csv(CSV_PATH)
print(f'Classi: {df["category"].nunique()}')
print(f'Fold: {sorted(df["fold"].unique())}')

In [ ]:
# CELLA 5 — Carica configurazione da YAML
import yaml

CONFIG_PATH = f'{REPO_DIR}/config/finetune.yaml'

with open(CONFIG_PATH) as f:
    yaml_config = yaml.safe_load(f)

CONFIG = {
    'num_classes':       yaml_config['model']['num_classes'],
    'drop_rate':         yaml_config['model']['drop_rate'],
    'n_epochs':          yaml_config['training']['n_epochs'],
    'batch_size':        yaml_config['training']['batch_size'],
    'num_workers':       yaml_config['training']['num_workers'],
    'val_ogni':          yaml_config['training']['val_ogni'],
    'lr_encoder':        yaml_config['optimizer']['lr_encoder'],
    'lr_classificatore': yaml_config['optimizer']['lr_classificatore'],
    'lr_min':            yaml_config['optimizer']['lr_min'],
    'weight_decay':      yaml_config['optimizer']['weight_decay'],
    'label_smoothing':   yaml_config['optimizer']['label_smoothing'],
}

print('Configurazione caricata:')
for k, v in CONFIG.items():
    print(f'  {k:<25} = {v}')

In [ ]:
# CELLA 6 — Test caricamento checkpoint
from src.model import MAEASTFineTune

print('Test caricamento checkpoint Patch...')
modello_test = MAEASTFineTune(
    checkpoint_path = CHECKPOINT_PATCH,
    num_classes     = CONFIG['num_classes'],
    drop_rate       = CONFIG['drop_rate'],
).to(DEVICE)

modello_test.eval()
x_test = torch.randn(2, 1, 128, 512).to(DEVICE)
with torch.no_grad():
    out = modello_test(x_test)

print(f'Input:  {x_test.shape}')
print(f'Output: {out.shape}  (atteso: [2, 50])')
assert out.shape == (2, 50)
print('Checkpoint Patch caricato correttamente')

del modello_test
torch.cuda.empty_cache()

In [ ]:
# CELLA 7A — Cross-validation modello PATCH
# Target paper: 90.0% | Tempo stimato: ~15-20 min su T4
import json
from src.train import cross_validation_5fold

print('MODELLO: MAE-AST Patch 12L (chunked masking 75%)')
print('Target paper: 90.0%')

risultati_patch = cross_validation_5fold(
    checkpoint_path = CHECKPOINT_PATCH,
    csv_path        = CSV_PATH,
    audio_dir       = AUDIO_DIR,
    config          = CONFIG,
    device          = DEVICE,
)

out_path = f'{RESULTS_DIR}/partB_patch.json'
with open(out_path, 'w') as f:
    json.dump({
        'modello':    'MAE-AST Patch 12L',
        'masking':    'chunked 75%',
        'target':     0.900,
        'accuracies': risultati_patch['accuracies'],
        'media':      risultati_patch['media'],
        'std':        risultati_patch['std'],
        'config':     CONFIG,
    }, f, indent=2)

print(f'Risultati salvati: {out_path}')

In [ ]:
# CELLA 7B — Cross-validation modello FRAME
# Target paper: 88.9% | Tempo stimato: ~15-20 min su T4
print('MODELLO: MAE-AST Frame 12L (random masking 75%)')
print('Target paper: 88.9%')

risultati_frame = cross_validation_5fold(
    checkpoint_path = CHECKPOINT_FRAME,
    csv_path        = CSV_PATH,
    audio_dir       = AUDIO_DIR,
    config          = CONFIG,
    device          = DEVICE,
)

out_path = f'{RESULTS_DIR}/partB_frame.json'
with open(out_path, 'w') as f:
    json.dump({
        'modello':    'MAE-AST Frame 12L',
        'masking':    'random 75%',
        'target':     0.889,
        'accuracies': risultati_frame['accuracies'],
        'media':      risultati_frame['media'],
        'std':        risultati_frame['std'],
        'config':     CONFIG,
    }, f, indent=2)

print(f'Risultati salvati: {out_path}')

In [ ]:
# CELLA 8 — Riepilogo e grafici
import matplotlib.pyplot as plt
import numpy as np

# Tabella
print('='*60)
print('  RIEPILOGO RISULTATI — Parte B')
print('='*60)
print(f'  {"Modello":<25} {"Nostro":>12} {"Paper":>8} {"Delta":>8}')
print('  ' + '-'*55)

for nome, risultati, target in [
    ('MAE-AST Patch 12L', risultati_patch, 0.900),
    ('MAE-AST Frame 12L', risultati_frame, 0.889),
]:
    media = risultati['media']
    std   = risultati['std']
    delta = (media - target) * 100
    segno = '+' if delta >= 0 else ''
    print(f'  {nome:<25} '
          f'{media*100:.1f}+/-{std*100:.1f}% '
          f'{target*100:.1f}% '
          f'{segno}{delta:.1f}%')

print('='*60)

# Grafico
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
x = np.arange(5)
w = 0.35
ax.bar(x - w/2, [a*100 for a in risultati_patch['accuracies']],
       w, label='Patch', color='#2E75B6', alpha=0.85)
ax.bar(x + w/2, [a*100 for a in risultati_frame['accuracies']],
       w, label='Frame', color='#70AD47', alpha=0.85)
ax.axhline(y=90.0, color='#2E75B6', linestyle='--', linewidth=1.5, label='Paper Patch')
ax.axhline(y=88.9, color='#70AD47', linestyle='--', linewidth=1.5, label='Paper Frame')
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(70, 100)
ax.set_title('Accuracy per fold — ESC-50')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
modelli = ['Paper\nPatch', 'Nostro\nPatch', 'Paper\nFrame', 'Nostro\nFrame']
valori  = [90.0, risultati_patch['media']*100,
           88.9, risultati_frame['media']*100]
colori  = ['#2E75B6', '#4472C4', '#70AD47', '#92D050']
bars = ax2.bar(modelli, valori, color=colori, alpha=0.85)
for bar, val in zip(bars, valori):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom')
ax2.set_ylim(75, 100)
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Confronto con il paper')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
graph_path = f'{RESULTS_DIR}/partB_risultati.png'
plt.savefig(graph_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Grafico salvato: {graph_path}')